# Running MD in the canoncial ensemble

In this notebook, we will discuss different types of thermodynamic ensembles with a focus on the Canonical ensemble. The exercise relies a lot on experimentation. It is sensible to open a document (presentation/word-like document etc.) to store figures.

## Microcanonical ensemble (NVE)

In this ensemble, the number of particles (N), system volume (V) and total energy (E) remain constant. This is what we did in the first MD example and is what is implemented via the velocity-Verlet algorithm. Since the total energy arises out of an interplay between kinetic and potential energy, we almost always need to introduce an outside interaction, e.g. with an external heat bath via a thermostat to equilibriate the structures before an NVE production run.

## Canonical ensemble (NVT) and thermostats

The canonical ensemble sets the number of particles, system volume and temperature (T) to constant values. In practice, this is realized via the means of a thermostat which adds forces to particles in a way that the system eventually reaches the target temperature. There are a number of realizations for this concept that differ in their properties regarding the time it takes to achieve equilibrium and ergodicity (ability to evenly explore the PES).

Most thermostats require a friction or dampening parameter that describes how strongly the external heat bath provided by the thermostat is "coupled" to the simulated system. The optimal choice depends on the thermostat in question and on the specific simulated system. One should be careful not to "overcouple" the thermostat where the added forces are so strong that the system reacts in unphysical ways. Due to this, it is desireable to only equilibriate a structure in an NVT ensemble and collect data regarding physical properties in the NVE ensemble. However, this is not possible for all applications.

In the following, we will discuss two specific thermostats that are commonly used. For coarse descriptions and instructions of other thermostats available in ASE, consult the [documentation](https://ase-lib.org/ase/md.html).

### Langevin thermostat

The Langevin thermostat achieves the desired temperature by adding two additional terms to the forces:

$\mathbf{F} = \mathbf{F}_c + \mathbf{F}_f + \mathbf{F}_r$,

$ \mathbf{F}_f = -m\gamma \mathbf{v}$,

$\mathbf{F}_r = \mathbf{R} \sqrt{k_B T m \lambda /dt}$.

Here, $F_c$ is the conventional force provided by either the model or DFT, $F_f$ is a frictional drag term that depends on the friction parameter $\gamma$, and $F_r$ is the random force term, which will be multiplied with a randomized direction vector $\mathbf{R}$. That last term drives the system towards its target temperature $T$. The choice of $\gamma$ is crucial as it adjusts the speed with which equilibrium is achieved. However, too high values can lead to unphysical behavior. This thermostat is relatively simple to implement and available in most MD codes. A drawback is its stochastic nature making it non-deterministic and the applied random direction vector frequently leads to a finite center-of-mass motion. One advantage of this implementation is that the implementation of multiple independent Langevin thermostats is straightforward making it a good choice for non-equilibrium simulations to e.g. simulate temperature gradients.

### Nosé-Hoover thermostat

The Nosé-Hoover thermostat is a commonly used deterministic method that adds a heat bath as an additional degree of freedom to the system. For mathematical details, see the [paper](https://journals.aps.org/pra/pdf/10.1103/PhysRevA.33.4253). A known issue is that ergodicity is not achieved even for simple systems such as an harmonic oscillator. This is why in practice several degrees of freedom are added in a "Nosé-Hoover chain". For the implementation in ASE, it is necessary to specify a characteristic time scale. This is the inverse of what we needed for the Langevin thermostat and a decent value is usually 100 times the time step. Higher values are safer but convergence will be reached more slowly. 

In [ ]:
from mace.calculators import MACECalculator
import ase.io
import ase.units
from pathlib import Path
import torch
import numpy as np

base_path = Path.cwd().parents[1]

model_file_name = (
    base_path
    / "data"
    / "models"
    / "full_dataset"
    / "M_2_ell_2_16_16_cut_5"
    / "Cu2S_stagetwo.model"
)
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"


# set up the calculator
calc = MACECalculator(model_file_name, device=device)

# set the structure file for Cu2S
struct_file_name = base_path / "data" / "Cu2S_FCC" / "Cu2S_conv.extxyz"

struct = ase.io.read(struct_file_name)

Now we set up the thermostat. We start with a Langevin thermostat with some basic settings.

In [ ]:
from ase.md.langevin import Langevin
from ase.build import make_supercell
from ase.md.velocitydistribution import (
    MaxwellBoltzmannDistribution,
    Stationary,
)

# we copy the structure to maintain the original
atoms = struct.copy()
# we make a 2x2x2 supercell (the difference in evaluation speed is small as for very small cells MACE needs to compute the 'extended' neighbor list up to the cutoff anyways)
atoms = make_supercell(atoms, np.diag([2, 2, 2]))

rng = np.random.default_rng(seed=27)
temperature = 300

# assign the calculator
atoms.calc = calc
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature, rng=rng)
Stationary(atoms)
# we increase the time step to accelerate the simulations. 2 fs is too large to obtain properly converged results for most systems
dt = 2.0
traj_path = (
    base_path
    / "data"
    / "MD_trajectories"
    / f"Cu2S_222_NVT_{temperature}K.traj"
)

# We set up the thermostat. The most important properties are the temperature and the friction coefficient.
integrator = Langevin(
    atoms,
    dt * ase.units.fs,
    temperature_K=temperature,
    friction=1e-2 / ase.units.fs,
    rng=rng,
    loginterval=10,
    logfile="-",
    trajectory=traj_path,
    fixcm=False,
)


integrator.run(500)

## Analyzing the trajectory

First let's observe the temperature trend with the thermostat.

In [ ]:
from ase.io.trajectory import Trajectory
from ase.visualize import view
import matplotlib.pyplot as plt


def running_average(y, window_size):
    kernel = np.ones(window_size) / window_size
    return np.convolve(y, kernel, mode="valid")


traj = Trajectory(traj_path, "r")
temperatures = [atoms.get_temperature() for atoms in traj]

plt.plot(np.arange(len(temperatures)) * dt, temperatures, label="raw data")
window = 10
ravg = running_average(temperatures, window)
plt.plot(
    np.arange(len(temperatures))[window // 2 - 1 : -window // 2] * dt,
    ravg,
    label="running average",
)
plt.xlabel("time / fs")
plt.ylabel("temperature / K")
plt.legend()
view(traj)

As we can see, the temperature does shows the same pronounced drop that we experienced previously. However, the thermostat raises it slowly. After some time, the temperature will arrive at 300 K. Yet, it will still show significant oscillations. Performing the run over a significant amount of time it should average out to the target temperature. Below there is a code snippet giving you the radial distribution functions for the trajectory. There, you can see the effect of the thermostat settings in a more quantitative manner.

We can technically accelerate convergence by adjusting the friction parameter.

TASK:
- Try changing the friction parameter to larger and smaller values and check the plot and trajectory. Do you see the constraining effect?
- Run the simulation for a bit longer to see if the temperature converges.
- Try higher temperatures - what do you see in the RDFs and MSDs? Is it harder to maintain a stable temperature?
- MLIPs such as MACE will struggle when conditions depart far from training data. This can happen for very high temperatures. Try a few temperatures and see when the system will become unstable.

For higher values, we should see a much faster convergence of the temperature and for smaller we will only barely reach the equilibrium. If the increase is substantial (e.g. to `1e-1`), the trajectory will be much more constrained and the atoms will only lightly oscillate around their equilibrium positions. This interference with the MD trajectory will lead to unphysical trajectories.
It is conventional to use a value of approximately 1% of the time step used. Depending on the specific application this would have to be adjusted.

TASK:
- Try a Nosé-Hoover thermostat: use `from ase.md.nose_hoover_chain import NoseHooverChainNVT`. Instead of setting a friction coefficient, you will have to set the inverse - a characteristic time scale. Use 100 times the time step. Check the docstring of the thermostat to figure out which parameters you don't need.


In [ ]:
# NH solution (collapsed)
from ase.md.nose_hoover_chain import NoseHooverChainNVT

atoms = struct.copy()
atoms = make_supercell(atoms, np.diag([2, 2, 2]))

rng = np.random.default_rng(seed=27)
temperature = 300

# assign the calculator
atoms.calc = calc
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature, rng=rng)
Stationary(atoms)

dt = 2.0
traj_path = (
    base_path
    / "data"
    / "MD_trajectories"
    / f"Cu2S_222_NVT_NH_{temperature}K.traj"
)
integrator = NoseHooverChainNVT(
    atoms,
    dt * ase.units.fs,
    temperature_K=temperature,
    tdamp=1e2 * ase.units.fs,
    loginterval=10,
    logfile="-",
    trajectory=traj_path,
    tchain=2,
)
integrator.run(500)

### Mean square displacement

The mean square displacement can give insights regarding which atoms move more in a given system. It is formed via the average squared deviation

$\text{MSD}(\tau) = \frac{1}{N} \sum_{i=1}^{N} \left\langle \left| \mathbf{r}_i(t + \tau) - \mathbf{r}_i(t) \right|^2 \right\rangle_t$.

In a crystal, this value is typically higher for lighter elements which oscillate more vividly around their equilibrium positions while for fluids the values tend to vary a lot more as convection occurs. In fact, for a 3D system, the MSD can be directly related to the diffusivity with

$D = \frac{1}{6} \lim_{\tau \to \infty} \frac{d}{d\tau} \text{MSD}(\tau)$.


In [ ]:
# IDEA WOULD BE: provide some code snippets for analysis here, let the participants try it out for a range of temperatures etc.

# MSD analysis
equilibration_time = 10
pos_array = np.asarray([atoms.positions for atoms in traj])
MSDs = {}
for unique_species in np.unique(struct.symbols):
    indices = np.where(struct.symbols == unique_species)[0]
    mean_pos = np.mean(pos_array[equilibration_time:], axis=0)
    MSDs[unique_species] = np.mean(
        (
            np.linalg.norm(
                pos_array[equilibration_time:, indices] - mean_pos[indices]
            )
        )
        ** 2
    )
    print(f"{unique_species} MSD: {MSDs[unique_species]}")

### Radial distribution function (RDF)

We will compute the RDFs between atoms in our trajectory.

$\mathrm{RDF} = \frac{V}{N^2} \sum_{i=1}^{N} \sum_{j \neq i}^{N} \delta(r - r_{ij})$

In [ ]:
import numpy as np
from ase.io import read


def compute_rdf(traj_file, species1, species2, r_max=10.0, n_bins=200, skip=0):
    frames = read(traj_file, index=f"{skip}:")

    dr = r_max / n_bins
    rdf = np.zeros(n_bins)
    n_frames = len(frames)

    for atoms in frames:
        symbols = np.asarray(atoms.symbols)

        idx1 = np.where(symbols == species1)[0]
        idx2 = np.where(symbols == species2)[0]

        # get distances within the minimum image convention
        dists = atoms.get_all_distances(mic=True)

        for i in idx1:
            idists = dists[i, idx2]
            # Exclude self (same atom)
            if species1 == species2:
                idists = idists[dists > 1e-8]
            hist, _ = np.histogram(idists, bins=n_bins, range=(0, r_max))
            rdf += hist

    r = np.linspace(dr / 2, r_max - dr / 2, n_bins)
    volume = atoms.get_volume()
    n2_density = len(np.where(symbols == species2)[0]) / volume

    shell_volumes = 4 * np.pi * r**2 * dr
    rdf /= n_frames * len(idx1) * n2_density * shell_volumes

    return r, rdf


sepcies1 = "Cu"
species2 = "S"

r, g_r = compute_rdf(traj_path, sepcies1, species2, r_max=8.0, skip=10)

import matplotlib.pyplot as plt

plt.plot(r, g_r)
plt.xlabel(r"r / $\mathrm{\AA}$")
plt.ylabel("RDF")
plt.title(f"{sepcies1}-{species2} RDF")

## Bonus task 1: Running NVE after NVT

TASK:
- try running an NVE simulation right after NVT, does it maintain the temperature once the hermostat is turned off?

If there are slight deviations, this might be because of fluctuations of the total energy due to the thermostat. If a very precise temperature is required one can calculate the average total energy during the equilibration and wait until the system reaches this total energy before switching from NVT to NVE. However, since oscillations of temperatures are usually less pronounced for the larger systems used in MD, the differences are usually very small.

In [ ]:
# NVE code (collapsed)
from ase.md.verlet import VelocityVerlet

velocities = []


integrator = VelocityVerlet(
    atoms,
    timestep=dt * ase.units.fs,
    loginterval=10,
    logfile="-",
)


def track_velocities(current_atoms=atoms, current_integrator=integrator):
    velocities.append(current_atoms.get_velocities())
    return True


integrator.attach(track_velocities, interval=1)

# run the simulation
n_steps = 1000
integrator.run(n_steps)

## Bonus task 2: correlation functions

In the following, we will compute the autocorrelation function of the velocities from the trajectory (preferably run in NVE). We will see a decay as a function of time with a certain oscillation profile. The frequencies corresponds to the vibrations in the system. If one takes the Fourier transform of this, we get the phonon density of states. This should also include the frequencies that you see with IR or Raman. However, just using this, we cannot identify which modes are either IR or Raman active. For this, we would have to investigate the dipole moment/polarizability, for which we need the electrons to some extent.

TASK:
- What does the resolution of the Spektrum depend on? Try changing the simulation settings to get a better vibrational profile, 

In [ ]:
from scipy.signal import correlate
from scipy.fft import fft, fftfreq

# VACF
velocities = np.array(velocities)

vacf = np.zeros(velocities.shape[0])
for aid in range(len(atoms)):
    for j in range(3):
        # Compute the autocorrelation function
        autocorrelation = correlate(
            velocities[:, aid, j], velocities[:, aid, j], mode="full"
        )[len(velocities[:, aid, j]) - 1 :]
        vacf += autocorrelation


plt.plot(
    np.arange(velocities.shape[0]) * dt, autocorrelation / autocorrelation[0]
)
plt.ylabel("VACF")
plt.xlabel("time / fs")
plt.figure(2)
plt.plot(
    fftfreq(len(autocorrelation))[: len(autocorrelation) // 2] * 1000 / dt,
    fft(autocorrelation / autocorrelation[0])[: len(autocorrelation) // 2],
)
plt.xlabel("frequency / THz")
plt.ylabel("phonon DOS")
plt.xlim([0, 15])